In [1]:
# !pip install basic_pitch
# !pip install pretty_midi

In [2]:
from basic_pitch.inference import predict
import librosa

model_output, midi_data, note_events = predict("./audio/test.mp3")
y, sr = librosa.load("./audio/test.mp3", sr=44100, mono=True)

Predicting MIDI for ./audio/test.mp3...


In [3]:
MIN_CONFIDENCE = 0.5
MIN_DURATION = 0.1

filtered = [
    n for n in note_events
    if n[2] > MIN_CONFIDENCE and (n[1]-n[0]) > MIN_DURATION
]
# note_events[0]

In [4]:
import pretty_midi

pm = pretty_midi.PrettyMIDI()
instrument = pretty_midi.Instrument(program=0)

for note in filtered:
    start = note[0]
    end   = note[1]
    pitch = note[2]
    confidence = note[3]
    note = pretty_midi.Note(
        velocity=int(confidence*127),
        pitch=int(pitch),
        start=start,
        end=end
    )
    instrument.notes.append(note)

pm.instruments.append(instrument)
pm.write("output/out_from_mp3.mid")

In [5]:
from utils.audio_cleanup import clean_notes

tempo, _ = librosa.beat.beat_track(y=y, sr=sr)

instrument.notes = clean_notes(
    instrument.notes,
    tempo=tempo,
    subdivision=4
)

[0.11319728]


In [6]:
pm.write("output/out_from_mp3.mid")